# 1. Đọc dữ liệu

In [81]:

df = pd.read_csv('../data/preproces_data.csv')
df.head()

,Diện tích đất:,Hướng cửa chính:,Giấy tờ pháp lý:,Đặc điểm nhà/đất:,Loại hình đất:,Chiều ngang:,Chiều dài:,Số phòng ngủ:,Số phòng vệ sinh:,Loại hình nhà ở:,...,Tầng số:,Hướng ban công:,Đặc điểm căn hộ:,Loại hình văn phòng:,district,city,Phân khu/Lô/Block/Tháp,has_floor_number,Price (trieu VND),log_price
0,170.0,Tây Bắc,Đã có sổ,Mặt tiền,Đất thổ cư,13.0,13.00,2.0,2.0,"Nhà mặt phố, mặt tiền",...,26.0,Đông Bắc,Căn góc,Shophouse,thành phố tân an,long an,Không thuộc project/block,1,419.0,6.040255
1,64.2,Tây Bắc,Đã có sổ,Nở hậu,Đất thổ cư,4.0,16.05,2.0,2.0,"Nhà mặt phố, mặt tiền",...,26.0,Đông Bắc,Căn góc,Shophouse,quận 9,tp hồ chí minh,Không thuộc project/block,1,936.0,6.842683
2,108.0,Tây,Đã có sổ,Nở hậu,Đất thổ cư,5.0,20.00,6.0,6.0,"Nhà ngõ, hẻm",...,26.0,Đông Bắc,Căn góc,Shophouse,quận 7,tp hồ chí minh,Không thuộc project/block,1,9500.0,9.159152
3,42.0,Nam,Đã có sổ,Nở hậu,Đất thổ cư,4.0,11.00,5.0,3.0,"Nhà ngõ, hẻm",...,26.0,Đông Bắc,Căn góc,Shophouse,quận 11,tp hồ chí minh,Không thuộc project/block,1,6500.0,8.779711
4,93.0,Nam,Đã có sổ,Hẻm xe hơi,Đất thổ cư,5.5,17.00,5.0,6.0,"Nhà ngõ, hẻm",...,26.0,Đông Bắc,Căn góc,Shophouse,quận bình tân,tp hồ chí minh,Không thuộc project/block,1,5000.0,8.517393


In [82]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5947 entries, 0 to 5946
Data columns (total 26 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Diện tích đất:            5947 non-null   float64
 1   Hướng cửa chính:          5947 non-null   object 
 2   Giấy tờ pháp lý:          5947 non-null   object 
 3   Đặc điểm nhà/đất:         5947 non-null   object 
 4   Loại hình đất:            5947 non-null   object 
 5   Chiều ngang:              5947 non-null   float64
 6   Chiều dài:                5947 non-null   float64
 7   Số phòng ngủ:             5947 non-null   float64
 8   Số phòng vệ sinh:         5947 non-null   float64
 9   Loại hình nhà ở:          5947 non-null   object 
 10  Tình trạng nội thất:      5947 non-null   object 
 11  Diện tích sử dụng:        5947 non-null   float64
 12  Tình trạng bất động sản:  5947 non-null   object 
 13  Diện tích:                5947 non-null   float64
 14  Loại hìn

# 2. Import thư viện sử dụng

Sử dụng 3 model:
 - lightgmb
 - catboost
 - xgboost

In [84]:
!pip install lightgbm xgboost catboost

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


In [85]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.preprocessing import MinMaxScaler, PolynomialFeatures, StandardScaler, Normalizer, LabelEncoder, OneHotEncoder
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
import xgboost as xgb

# 1. One-hot endcoding với các 

In [86]:
object_cols = df.loc[:, df.dtypes == object].columns # tìm các cột kiểu dữ liệu object trong dataframe

In [102]:
df_temp = df.copy()
onehot_dict = {}

for col in object_cols:
    enc = OneHotEncoder(sparse_output=False)  # bỏ luôn .toarray()
    transformed = enc.fit_transform(df_temp[col].values.reshape(-1, 1))
    onehot_dict[col] = enc
    
    # Đặt tên cột theo tên cột gốc để tránh trùng
    feature_names = [f"{col}_{cat}" for cat in enc.categories_[0]]
    ohe_df = pd.DataFrame(transformed, columns=feature_names, index=df_temp.index)
    
    df_temp = df_temp.drop([col], axis=1)
    df_temp = df_temp.join(ohe_df)

In [103]:
df_temp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5947 entries, 0 to 5946
Columns: 429 entries, dien_tich_dat to phan_khu_lo_block_thap_W3
dtypes: float64(428), int64(1)
memory usage: 19.5 MB


In [106]:
print("Số cột sau One-Hot:", df_temp.shape[1])

Số cột sau One-Hot: 429


Sau khi One-Hot Endcoding, tổng quan kết quả:
1) 429 columns: từ 29 columns ban đầu đã chuyển đổi thành 429 cột vì mỗi giá tị unique của cột chữ thành 1 cột riêng
2) 5947 mẫu dữ liệu (đáp ứng yêu cầu - tối thiểu 5000)
3) dtype gồm 2 loại: 
- float64 - 428 columns
- int64 - 1 columns

Tuy nhiên hiện tại các name column không phải ở dạng unidecode, nên XGBoost không xử lý được, cần chuyển về unidecode để phục vụ việc train data

In [108]:
from unidecode import unidecode
import re

def clean_col(col):
    col = unidecode(col)
    col = re.sub(r'[^A-Za-z0-9_]', '_', col)
    return col

df_temp.columns = [clean_col(col) for col in df_temp.columns]

print("Tên cột sau clean:", df_temp.columns.tolist()[:429])

Tên cột sau clean: ['dien_tich_dat', 'chieu_ngang', 'chieu_dai', 'so_phong_ngu', 'so_phong_ve_sinh', 'dien_tich_su_dung', 'dien_tich', 'tong_so_tang', 'tang_so', 'has_floor_number', 'price_trieu_vnd', 'log_price', 'huong_cua_chinh_Bac', 'huong_cua_chinh_Nam', 'huong_cua_chinh_Tay', 'huong_cua_chinh_Tay_Bac', 'huong_cua_chinh_Tay_Nam', 'huong_cua_chinh_Dong', 'huong_cua_chinh_Dong_Bac', 'huong_cua_chinh_Dong_Nam', 'giay_to_phap_ly_Giay_to_khac', 'giay_to_phap_ly_Dang_cho_so', 'giay_to_phap_ly_Da_co_so', 'dac_diem_nha_dat_Hem_xe_hoi', 'dac_diem_nha_dat_Mat_tien', 'dac_diem_nha_dat_No_hau', 'loai_hinh_dat_Dat_cong_nghiep', 'loai_hinh_dat_Dat_nong_nghiep', 'loai_hinh_dat_Dat_nen_du_an', 'loai_hinh_dat_Dat_tho_cu', 'loai_hinh_nha_o_Nha_biet_thu', 'loai_hinh_nha_o_Nha_mat_pho__mat_tien', 'loai_hinh_nha_o_Nha_ngo__hem', 'loai_hinh_nha_o_Nha_pho_lien_ke', 'tinh_trang_noi_that_Ban_giao_tho', 'tinh_trang_noi_that_Hoan_thien_co_ban', 'tinh_trang_noi_that_Noi_that_cao_cap', 'tinh_trang_noi_that_No

# 4. Chia tập train/test 

In [109]:
X = df_temp.drop('price_trieu_vnd', axis=1)
Y = df_temp['price_trieu_vnd']

In [110]:
X

,dien_tich_dat,chieu_ngang,chieu_dai,so_phong_ngu,so_phong_ve_sinh,dien_tich_su_dung,dien_tich,tong_so_tang,tang_so,has_floor_number,...,phan_khu_lo_block_thap_S1,phan_khu_lo_block_thap_S4,phan_khu_lo_block_thap_S6,phan_khu_lo_block_thap_S7,phan_khu_lo_block_thap_SAI_GON_VILLAGE,phan_khu_lo_block_thap_T28,phan_khu_lo_block_thap_T5,phan_khu_lo_block_thap_THANG_LONG_CITY,phan_khu_lo_block_thap_TAN_KIEN_RESIDENCES,phan_khu_lo_block_thap_W3
0,170.00,13.0,13.00,2.0,2.0,40.0,103.0,1.0,26.0,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,64.20,4.0,16.05,2.0,2.0,40.0,103.0,1.0,26.0,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,108.00,5.0,20.00,6.0,6.0,226.0,103.0,2.0,26.0,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,42.00,4.0,11.00,5.0,3.0,226.0,103.0,2.0,26.0,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,93.00,5.5,17.00,5.0,6.0,226.0,103.0,5.0,26.0,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5942,33.00,6.0,11.00,4.0,3.0,31.0,35.0,4.0,18.0,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5943,56.00,4.0,14.50,3.0,2.0,96.0,35.0,4.0,18.0,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5944,68.00,4.0,17.00,3.0,2.0,96.0,35.0,4.0,18.0,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5945,68.00,4.0,17.00,1.0,1.0,96.0,36.0,4.0,18.0,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [111]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X,Y, test_size = 0.3, random_state = 42
)

# 5. Huấn luyện mô hình

In [112]:

pipelines = {
    'linear': [('model', LinearRegression())],
    'xgboost': [('model',xgb.XGBRegressor(
    objective='reg:squarederror',  # for regression
    ))],
    'histogram': [('model',HistGradientBoostingRegressor())],
    'catboost': [('model',CatBoostRegressor())],
    'SVR_linear': [('model',SVR(kernel='linear'))],
    'SVR_poly': [('model',SVR(kernel='poly'))],
    'SVR_rbf': [('model',SVR(kernel='rbf'))],
    'lightGBM': [('model',LGBMRegressor())]
}

In [115]:
results = []
for name, pipeline in pipelines.items():
    model = Pipeline(pipeline).fit(X_train.values, Y_train.values)
    Y_pred = model.predict(X_test.values)
    r2 = r2_score(Y_test, Y_pred)
    mse = mean_squared_error(Y_test, Y_pred)
    rmse = np.sqrt(mean_squared_error(Y_test, Y_pred))
    mae = mean_absolute_error(Y_test, Y_pred)
    results.append({
        'model': name,
        'r2': r2,
        'mse': mse,
        'rmse': rmse,
        'mae': mae,
        'model_train': model
    })

Learning rate set to 0.05129
0:	learn: 3192.6363014	total: 6.1ms	remaining: 6.1s
1:	learn: 3056.0911465	total: 8.49ms	remaining: 4.24s
2:	learn: 2927.6935982	total: 11.5ms	remaining: 3.81s
3:	learn: 2797.7499355	total: 13.8ms	remaining: 3.43s
4:	learn: 2684.6924120	total: 16ms	remaining: 3.17s
5:	learn: 2568.9294819	total: 18.4ms	remaining: 3.04s
6:	learn: 2456.7183798	total: 21.1ms	remaining: 2.99s
7:	learn: 2354.3088627	total: 23.4ms	remaining: 2.9s
8:	learn: 2251.0831884	total: 25.8ms	remaining: 2.84s
9:	learn: 2153.9488409	total: 28.5ms	remaining: 2.82s
10:	learn: 2055.4962017	total: 30.5ms	remaining: 2.74s
11:	learn: 1975.7973060	total: 32.5ms	remaining: 2.68s
12:	learn: 1892.2314150	total: 34.3ms	remaining: 2.6s
13:	learn: 1808.9820541	total: 36.6ms	remaining: 2.58s
14:	learn: 1737.8304292	total: 38.4ms	remaining: 2.52s
15:	learn: 1662.7804375	total: 40.3ms	remaining: 2.48s
16:	learn: 1589.0270554	total: 42.5ms	remaining: 2.46s
17:	learn: 1519.1837918	total: 44.7ms	remaining: 2.4

/Users/nals_macbook_108/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [116]:
pd.DataFrame(results).sort_values(by=['r2', 'mse'],
                                  ascending=[False, True]).reset_index(drop=True)

,model,r2,mse,rmse,mae,model_train
0,catboost,0.999190,8.840423e+03,94.023524,32.801286,(CatBoostRegressor(loss_function='RMSE'))
1,lightGBM,0.999052,1.034396e+04,101.705257,16.942697,(LGBMRegressor())
2,xgboost,0.999034,1.054266e+04,102.677439,26.606473,"(XGBRegressor(base_score=None, booster=None, c..."
3,histogram,0.996580,3.731897e+04,193.181177,34.980749,(HistGradientBoostingRegressor())
4,SVR_linear,0.510094,5.345506e+06,2312.035097,1077.167490,(SVR(kernel='linear'))
5,SVR_rbf,-0.078482,1.176763e+07,3430.397493,2075.170253,(SVR())
6,SVR_poly,-0.078566,1.176855e+07,3430.531587,2075.283187,(SVR(kernel='poly'))
7,linear,-1.830267,3.088186e+07,5557.145221,1270.198116,(LinearRegression())


### Metric đánh giá 
Metric r2: tỉ lệ phương sai dữ liệu mà mô hình giải thchs được
- r2 = 1.0 --> hoàn hảo, r2 = 0 --> không tốt hơn dự đoán trung bình, r2 < 0 --> tệ hơn dự đoán trung bình
Metric rmse/mse: sai số bình phương - phạt năng các lỗi lớn
Metric mae: sai số tuyệt đối trung bình - ít bị ảnh hưởng bởi outlier
### Phân tích chi tiết
- Nhóm gradient boosting (Mô hình catboost, lightGBM, XGboost, histogram)) hoạt động tốt với r2 = 0.999, nghĩa là mô hình giải thích được > 99,9% biến động của dữ liệu, phù hợp với dữ liệu cấu trúc phi tuyến
- SVR Linear có r2 = 0.51 chỉ giải thích được ~ 51% biến động của dữ liệu, cho thấy dữ liệu tính phi tuyến mạnh, kernel tuyến tính không đủ phức tạp
- SVR_rbf, SVR_poly và Linear Regression có r2 <0 - tệ, nguyên nhân:
+ dữ liệu chưa được chuẩn hoá (scaling) - SVR và Linear Regression rất nhạy cảm với scale features
+ dữ liệu có thể có ôutpier lớn ảnh hưởng nặng

--> Phân tích tiếpnhóm Gradient Boosting, tốt nhất sử dụng model Catboost làm mô hình chính vì r2 cao nhất và mae thấp nhất (32.8).